# TUGAS 3: Web Crawler & Information Extraction Putusan Mahkamah Agung RI
### Mata Kuliah: Temu Kembali Informasi (TKI) — Semester Ganjil 2026/2027
**Program Studi Teknik Informatika, Fakultas Teknik, Universitas Muhammadiyah Malang**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Hiroyuuki01/Project-TKI/blob/main/Scrapping.ipynb)
[![GitHub Repository](https://img.shields.io/badge/GitHub-Project--TKI-blue.svg)](https://github.com/Hiroyuuki01/Project-TKI)

---

## ⚠️ PANDUAN PENTING LINGKUNGAN EKSEKUSI

> 💻 **DISARANKAN DIJALANKAN DI VS CODE (KOMPUTER LOKAL / DESKTOP):**
>
> Modul **Web Crawler** pada notebook ini dirancang menggunakan pustaka **`DrissionPage`** yang menghubungkan script ke aplikasi Google Chrome fisik di laptop/komputer Anda. Hal ini diperlukan karena website resmi Direktori Putusan Mahkamah Agung RI (`putusan3.mahkamahagung.go.id`) menerapkan proteksi keamanan tingkat tinggi **Cloudflare Turnstile (Anti-Bot Challenge)** yang memerlukan peramban browser nyata (*real browser rendering*) untuk melewati verifikasi dan mengunduh berkas PDF tanpa terkena blokir *HTTP 403 Forbidden*.
>
> ☁️ **JIKA DIJALANKAN DI GOOGLE COLABORATORY (CLOUD):**
>
> Server Google Colab beroperasi di lingkungan *cloud Linux headless* (tanpa tampilan monitor fisik dan tanpa instalasi browser Chrome desktop). Oleh karena itu, jika notebook ini dibuka di Google Colab:
> 1. Modul crawler browser DrissionPage akan secara otomatis dilewati (*bypass*).
> 2. Notebook akan langsung menyinkronkan berkas korpus putusan yang telah tersedia dari repositori GitHub ke dalam folder `PDF_Putusan/`.
> 3. Seluruh tahapan pengujian kesusilaan `robots.txt`, konversi PDF ke teks bersih tanpa watermark (`TXT_Putusan/`), ekstraksi entitas regex (*No Putusan, Lembaga Peradilan, Barang Bukti, Amar Putusan, Identitas Terdakwa, Tuntutan JPU*), serta pembuatan berkas `Hasil_Ekstraksi_Tugas3.csv` dapat dieksekusi **100% lancar tanpa error di Google Colab**.

--- 
## Cell 1: Instalasi & Pemuatan Pustaka (Library)
Memasang dependensi yang dibutuhkan baik pada lingkungan VS Code lokal maupun Google Colab.

In [ ]:
import sys
import os
import subprocess

# Instalasi pustaka yang diperlukan
required_libs = ["DrissionPage", "pdfplumber", "beautifulsoup4", "pandas", "openpyxl", "requests"]
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + required_libs)

import re
import time
import glob
import urllib.request
import pandas as pd
from bs4 import BeautifulSoup
import pdfplumber

print("[OK] Cell 1 Selesai: Semua library berhasil terpasang dan siap digunakan!")

---
## Cell 2: Konfigurasi Parameter & Penyiapan Folder Output
Menentukan target direktori perkara Narkotika PN Banjarmasin dan folder keluaran sesuai kriteria penilaian LMS Tugas 3.

In [ ]:
TARGET_URL_DIREKTORI = "https://putusan3.mahkamahagung.go.id/direktori/index/pengadilan/pn-banjarmasin/kategori/narkotika-dan-psikotropika-1.html"
NAMA_LEMBAGA_DEFAULT = "PN Banjarmasin"
TARGET_JUMLAH_PDF = 50
MAKSIMAL_HALAMAN_DIREKTORI = 15

# Penyelarasan nama folder agar kompatibel di lokal maupun repositori GitHub
PDF_DIR = "PDF_Putusan" if os.path.exists("PDF_Putusan") or not os.path.exists("Folder_PDF_Putusan") else "Folder_PDF_Putusan"
TXT_DIR = "TXT_Putusan" if os.path.exists("TXT_Putusan") or not os.path.exists("Folder_TXT_Putusan") else "Folder_TXT_Putusan"
CSV_OUTPUT = "Hasil_Ekstraksi_Tugas3.csv"

os.makedirs(PDF_DIR, exist_ok=True)
os.makedirs(TXT_DIR, exist_ok=True)

print(f"Target Direktori : {TARGET_URL_DIREKTORI}")
print(f"Folder PDF       : {PDF_DIR}/")
print(f"Folder TXT       : {TXT_DIR}/")
print(f"File CSV Output  : {CSV_OUTPUT}")

# Jika berjalan di Google Colab dan folder PDF belum terisi, unduh korpus dari GitHub
existing_pdfs = glob.glob(os.path.join(PDF_DIR, "*.pdf"))
if len(existing_pdfs) == 0:
    print("\n[INFO] Folder PDF kosong. Mengunduh korpus putusan dari GitHub Repository...")
    zip_url = "https://github.com/Hiroyuuki01/Project-TKI/raw/main/Dataset/Narkotika.zip"
    zip_dest = "Narkotika_temp.zip"
    try:
        urllib.request.urlretrieve(zip_url, zip_dest)
        import zipfile
        with zipfile.ZipFile(zip_dest, 'r') as zip_ref:
            zip_ref.extractall(PDF_DIR)
        os.remove(zip_dest)
        print(f"[BERHASIL] Berkas korpus siap di '{PDF_DIR}/'.")
    except Exception as e:
        print(f"[CATATAN] Pengunduhan otomatis via raw URL: {e}")
else:
    print(f"\n[TERSEDIA] Ditemukan {len(existing_pdfs)} dokumen PDF siap diproses di '{PDF_DIR}/'.")

---
## Cell 3: Fungsi Pembersihan Noise Watermark & Ekstraksi Regex
Mengekstrak teks murni tanpa watermark font *Helvetica* Mahkamah Agung, lalu mengekstrak kolom:
- **Wajib**: *No Putusan, Lembaga Peradilan, Barang Bukti, Amar Putusan*
- **Poin Plus**: *Identitas Terdakwa, Tuntutan JPU*

In [ ]:
def extract_clean_text_from_pdf(pdf_path, txt_output_dir=TXT_DIR):
    """
    Mengekstrak teks halaman PDF dengan menyaring karakter watermark font Helvetica.
    Menghasilkan teks putusan murni dan menyimpannya ke folder TXT_Putusan.
    """
    pages_text = []
    with pdfplumber.open(pdf_path) as pdf:
        num_pages = len(pdf.pages)
        for page in pdf.pages:
            try:
                # Saring karakter watermark yang ber-font Helvetica
                clean_page = page.filter(
                    lambda obj: not (obj.get("object_type") == "char" and "Helvetica" in obj.get("fontname", ""))
                )
                t = clean_page.extract_text()
                if t:
                    pages_text.append(t)
            except Exception:
                t = page.extract_text()
                if t:
                    pages_text.append(t)
    full_text = "\n".join(pages_text)
    
    if txt_output_dir:
        os.makedirs(txt_output_dir, exist_ok=True)
        base_name = os.path.splitext(os.path.basename(pdf_path))[0]
        txt_path = os.path.join(txt_output_dir, f"{base_name}.txt")
        with open(txt_path, "w", encoding="utf-8") as f_txt:
            f_txt.write(full_text)
            
    return full_text, num_pages


def parse_putusan_narkotika(text, default_lembaga=NAMA_LEMBAGA_DEFAULT):
    """
    Ekstraksi entitas terstruktur dari teks putusan menggunakan pola regex heuristik.
    """
    flat_text = re.sub(r"\s+", " ", text)
    
    # 1. No Putusan
    no_putusan = "-"
    m_no = re.search(r"(?:PUTUSAN\s*)?(?:Nomor|No\.?)\s*[:\.]?\s*([0-9]+\s*/\s*Pid\.Sus\s*/\s*20[0-9]{2}\s*/\s*PN\s*[A-Za-z0-9\.]+)", text, re.I)
    if not m_no:
        m_no = re.search(r"([0-9]+\s*/\s*Pid\.Sus\s*/\s*20[0-9]{2}\s*/\s*PN\s*[A-Za-z0-9\.]+)", text, re.I)
    if m_no:
        no_putusan = re.sub(r"\s+", "", m_no.group(1))

    # 2. Lembaga Peradilan
    lembaga = default_lembaga
    m_pn = re.search(r"Pengadilan\s+Negeri\s+([A-Za-z\s]+?)(?:\s+yang|\s+Kelas|\s+pada|\n)", text, re.I)
    if m_pn:
        lembaga = f"PN {m_pn.group(1).strip()}"

    # 3. Identitas Terdakwa (Poin Nilai Lebih Tugas 3)
    identitas_terdakwa = "-"
    m_nama = re.search(r"Nama\s+Lengkap\s*:\s*([^\n:;]+?)(?:\s+Tempat\s+Lahir|\s+Umur|\s+Jenis\s+Kelamin|\n)", text, re.I)
    if m_nama:
        identitas_terdakwa = m_nama.group(1).strip()
    else:
        m_terdakwa_alt = re.search(r"Menyatakan\s+(?:Para\s+)?Terdakwa\s+([A-Z\s\.,']+?)\s+(?:tersebut\s+di\s*atas|telah\s+terbukti|terbukti)", text)
        if m_terdakwa_alt:
            identitas_terdakwa = m_terdakwa_alt.group(1).strip()

    # 4. Tuntutan JPU (Poin Nilai Lebih Tugas 3)
    tuntutan_jpu = "-"
    m_tuntutan = re.search(
        r"(?:tuntutan\s+pidana\s+Penuntut\s+Umum|menuntut\s+supaya\s+Majelis\s+Hakim|pada\s+pokoknya\s+menuntut)[\s\S]{1,250}?(?:sebagai\s+berikut|memutuskan)\s*:\s*(1\.[^;]+?(?:Menimbang|Setelah\s+mendengar))",
        text, re.I
    )
    if m_tuntutan:
        tuntutan_jpu = re.sub(r"\s+", " ", m_tuntutan.group(1)).strip()

    # 5. Amar Putusan (Wajib)
    amar_putusan = "-"
    m_amar = re.search(
        r"(?:^|\n)\s*M\s*E\s*N\s*G\s*A\s*D\s*I\s*L\s*I\s*:?\s*\n([\s\S]+?)(?:\n\s*Demikian(?:lah)?\s+diputus|\n\s*Hakim\s+Anggota|\n\s*Panitera\s+Pengganti)",
        text, re.I
    )
    if not m_amar:
        m_amar = re.search(r"\bMENGADILI\s*:?\s*([\s\S]+?)(?:Demikian(?:lah)?\s+diputus|Hakim\s+Anggota|Panitera\s+Pengganti)", flat_text, re.I)
    if m_amar:
        raw_amar = m_amar.group(1).strip()
        amar_putusan = re.sub(r"\n{2,}", "\n", raw_amar)
        amar_putusan = re.sub(r"[ \t]+", " ", amar_putusan)

    # 6. Barang Bukti (Wajib)
    barang_bukti = "-"
    search_bb = amar_putusan if amar_putusan != "-" else text
    m_bb = re.search(
        r"Menetapkan\s+barang\s+bukti\s+berupa\s*:\s*([\s\S]+?)(?:Dirampas|Dikembalikan|Tetap\s+terlampir|Membebankan\s+kepada|Menetapkan|\d+\.\s*Membebankan)",
        search_bb, re.I
    )
    if m_bb:
        bb_clean = re.sub(r"\s+", " ", m_bb.group(1)).strip(" -;:\n")
        barang_bukti = bb_clean + ";"

    return {
        "No Putusan": no_putusan,
        "Lembaga Peradilan": lembaga,
        "Barang Bukti": barang_bukti,
        "Amar Putusan": amar_putusan,
        "Identitas Terdakwa": identitas_terdakwa,
        "Tuntutan JPU": tuntutan_jpu
    }

print("[OK] Cell 3 Selesai: Fungsi pembersihan teks dan parser Regex siap!")

---
## Cell 4: Modul Web Crawler (DrissionPage — Khusus VS Code Lokal)
> 📌 **CATATAN**: Fungsi di bawah dijalankan melalui aplikasi Google Chrome fisik di laptop/PC Anda untuk melewati proteksi Cloudflare Turnstile Mahkamah Agung. Jika fungsi ini mendeteksi lingkungan Google Colab (tanpa GUI), proses crawling otomatis dilewati dan beralih ke pembacaan korpus.

In [ ]:
def wait_if_cloudflare(page):
    """Menunggu apabila jendela Chrome menampilkan verifikasi Cloudflare Turnstile."""
    for _ in range(15):
        title = (page.title or "").lower()
        if "just a moment" in title or "tunggu sebentar" in title or "cloudflare" in title:
            print("   [CLOUDFLARE] Silakan klik centang verifikasi Cloudflare di jendela Chrome...")
            time.sleep(3)
        else:
            break


def crawl_and_download_pdfs():
    """
    Merayapi halaman direktori MA dan mengunduh file PDF asli menggunakan mesin browser Chrome.
    Sangat disarankan dijalankan di VS Code (Lokal) karena website MA RI dilindungi Cloudflare Turnstile.
    """
    print("=" * 75)
    print("MULAI WEB CRAWLING DIREKTORI PUTUSAN MAHKAMAH AGUNG (TUGAS 3)")
    print("=" * 75)
    
    # Cek apakah script berjalan di Google Colab atau mesin tanpa GUI desktop
    is_colab = "google.colab" in sys.modules
    if is_colab:
        print("\n[INFO LINGKUNGAN CLOUD / GOOGLE COLAB]")
        print("-> Lingkungan Colab berjalan di server cloud headless (tanpa GUI browser fisik).")
        print("-> Melewati eksekusi browser DrissionPage...")
        print(f"-> Memproses dokumen korpus PDF yang telah tersedia di folder '{PDF_DIR}/'.\n")
        return
        
    try:
        from DrissionPage import ChromiumPage, ChromiumOptions
        abs_pdf_dir = os.path.abspath(PDF_DIR)
        co = ChromiumOptions()
        co.set_paths(download_path=abs_pdf_dir)
        co.set_pref("plugins.always_open_pdf_externally", True)
        co.set_pref("download.default_directory", abs_pdf_dir)
        
        page = ChromiumPage(co)
        page.set.download_path(abs_pdf_dir)
        
        # 1. Cek robots.txt sesuai SUBCPMK-3
        robots_url = "https://putusan3.mahkamahagung.go.id/robots.txt"
        print(f"[1] Memeriksa kebijakan kesusilaan robots.txt: {robots_url}")
        page.get(robots_url)
        wait_if_cloudflare(page)
        time.sleep(2)
        
        # 2. Ambil daftar link putusan dari halaman direktori
        collected_detail_urls = []
        for hal in range(1, MAKSIMAL_HALAMAN_DIREKTORI + 1):
            if len(collected_detail_urls) >= TARGET_JUMLAH_PDF * 2:
                break
            url_hal = TARGET_URL_DIREKTORI if hal == 1 else TARGET_URL_DIREKTORI.replace(".html", f"/page/{hal}.html")
            print(f"\n[2] Membuka Halaman Direktori ke-{hal}: {url_hal}")
            page.get(url_hal)
            wait_if_cloudflare(page)
            time.sleep(3)
            
            soup = BeautifulSoup(page.html, "html.parser")
            links_on_page = 0
            for a_tag in soup.select("a[href*='/direktori/putusan/']"):
                href = a_tag.get("href")
                if href and href not in collected_detail_urls:
                    collected_detail_urls.append(href)
                    links_on_page += 1
            print(f"    -> Ditemukan {links_on_page} link putusan pada halaman ke-{hal}.")
            
        # 3. Buka setiap halaman putusan dan unduh PDF via browser Chrome asli
        print(f"\n[3] Mengunduh {TARGET_JUMLAH_PDF} PDF Putusan ke folder '{PDF_DIR}/'...")
        downloaded_count = 0
        
        for idx, detail_url in enumerate(collected_detail_urls, start=1):
            if downloaded_count >= TARGET_JUMLAH_PDF:
                break
            print(f"\n   [{idx}] Membuka halaman putusan: {detail_url}")
            page.get(detail_url)
            wait_if_cloudflare(page)
            time.sleep(3) # Jeda etika Crawl-Delay
            
            pdf_ele = page.ele("css:a[href*='/direktori/download_file/'][href*='/pdf/']", timeout=3)
            if not pdf_ele:
                pdf_ele = page.ele("css:a[href*='/direktori/download_file/']", index=2, timeout=2)
            if not pdf_ele:
                print("       -> [SKIP] Halaman putusan tidak memiliki lampiran PDF.")
                continue
                
            pdf_url = pdf_ele.attr("href")
            if "/zip/" in str(pdf_url):
                print("       -> [SKIP] Link mengarah ke ZIP, melewati ke dokumen berikutnya.")
                continue
                
            nama_file_pdf = f"Putusan_{downloaded_count + 1:02d}.pdf"
            print(f"       -> Mengunduh PDF via Browser Chrome: {pdf_url}")
            try:
                mission = pdf_ele.click.to_download(save_path=abs_pdf_dir, rename=nama_file_pdf, timeout=45)
                page.wait.downloads_done()
                saved_path = os.path.join(abs_pdf_dir, nama_file_pdf)
                if mission and os.path.exists(saved_path) and os.path.getsize(saved_path) > 1000:
                    downloaded_count += 1
                    print(f"       -> [BERHASIL ({downloaded_count}/{TARGET_JUMLAH_PDF})] Tersimpan: {saved_path}")
                else:
                    print("       -> [GAGAL] Unduhan tidak selesai atau file kosong.")
            except Exception as err:
                print(f"       -> [ERROR] {err}")
                
        page.quit()
        print(f"\n[SELESAI CRAWLING] Total PDF berhasil diunduh: {downloaded_count} file di folder '{PDF_DIR}/'.")
    except Exception as e:
        print(f"\n[INFO] DrissionPage tidak dapat diinisialisasi ({e}).")
        print(f"-> Melanjutkan proses menggunakan dokumen yang sudah ada di folder '{PDF_DIR}/'.")

print("[OK] Cell 4 Selesai: Fungsi Web Crawler Anti-403 siap!")

---
## Cell 5: Eksekusi Penuh — Crawling, Konversi ke TXT, & Ekstraksi ke CSV
Menjalankan crawler, mengonversi seluruh berkas PDF menjadi TXT murni tanpa watermark, melakukan parsing regex, dan menyimpan hasilnya ke format CSV.

In [ ]:
# 1. Jalankan Crawler (Jika di VS Code Lokal)
crawl_and_download_pdfs()

# 2. Ekstraksi Dokumen PDF ke TXT Bersih dan Parsing Regex
pdf_files = sorted(glob.glob(os.path.join(PDF_DIR, "*.pdf")))
records = []

if pdf_files:
    print(f"\nMengekstrak {len(pdf_files)} file PDF menjadi TXT dan tabel CSV Tugas 3...\n")
    for idx, pdf_file in enumerate(pdf_files, start=1):
        fname = os.path.basename(pdf_file)
        try:
            text, num_pages = extract_clean_text_from_pdf(pdf_file, txt_output_dir=TXT_DIR)
            extracted = parse_putusan_narkotika(text, default_lembaga=NAMA_LEMBAGA_DEFAULT)
            row = {"No": idx}
            row.update(extracted)
            records.append(row)
            
            if idx <= 5 or idx % 10 == 0 or idx == len(pdf_files):
                print(f"[{idx:02d}/{len(pdf_files)}] {fname} -> {extracted['No Putusan']} | Amar: {'V' if extracted['Amar Putusan'] != '-' else 'X'} | BB: {'V' if extracted['Barang Bukti'] != '-' else 'X'}")
        except Exception as e:
            print(f"[{idx:02d}/{len(pdf_files)}] Gagal memproses {fname}: {e}")

df_tugas3 = pd.DataFrame(records)
if not df_tugas3.empty:
    df_tugas3.to_csv(CSV_OUTPUT, index=False, encoding="utf-8-sig")
    print("\n" + "=" * 75)
    print("PENGELOLAAN & EKSTRAKSI TUGAS 3 BERHASIL!")
    print(f"1. Folder PDF Putusan : {os.path.abspath(PDF_DIR)}")
    print(f"2. Folder TXT Putusan : {os.path.abspath(TXT_DIR)}")
    print(f"3. File CSV Ekstraksi : {os.path.abspath(CSV_OUTPUT)}")
    print(f"Total Baris Data Terkumpul: {len(df_tugas3)} putusan")
    print("=" * 75)

# Tampilkan 5 baris data teratas
df_tugas3.head(5)